# Environment Setup for Azure AI Foundry Workshop

This notebook will guide you through setting up your environment for the Azure AI Foundry workshop.

## Prerequisites
- Python 3.8 or later
- Azure subscription with AI services access
- Basic Python knowledge

## Azure Authentication Setup
First, we'll verify our Azure credentials and setup.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType
import os

# Initialize Azure credentials
try:
    credential = DefaultAzureCredential()
    print("✓ Successfully initialized DefaultAzureCredential")
except Exception as e:
    print(f"× Error initializing credentials: {str(e)}")

## Initialize AI Project Client

> **Note:** Before proceeding, ensure you have set the following in your `.env` file:
>
> | Variable | Where to find it |
> |---|---|
> | `PROJECT_ENDPOINT` | Azure AI Foundry portal → your project → **Overview** → *Project endpoint* |
> | `AZURE_SUBSCRIPTION_ID` | Azure portal → Subscriptions, or the project Overview page |
> | `AZURE_RESOURCE_GROUP` | Azure portal → your Foundry hub → **Overview** → *Resource group* |
> | `AZURE_AI_PROJECT_NAME` | Your project name (also the last segment of the endpoint URL) |

Your `.env` should look like:
```
PROJECT_ENDPOINT=https://<hub-name>.services.ai.azure.com/api/projects/<project-name>
AZURE_SUBSCRIPTION_ID=xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
AZURE_RESOURCE_GROUP=my-resource-group
AZURE_AI_PROJECT_NAME=my-project-name
```

## Understanding AIProjectClient

The AIProjectClient is a key component for interacting with Azure AI services that:

- **Manages Connections**: Lists and accesses Azure AI resources like OpenAI models
- **Handles Authentication**: Securely connects using Azure credentials  
- **Enables Model Access**: Provides interfaces to use AI models and deployments
- **Manages Project Settings**: Controls configurations for your Azure AI project

The client requires:
- A project connection string (from Azure AI project settings)
- Valid Azure credentials

You can find your project connection string in Azure AI Studio under Project Settings:



In [ ]:
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse
import requests

# ── Load .env ────────────────────────────────────────────────────────────────
notebook_path = Path().absolute()
env_path = notebook_path.parent / '.env'
loaded = load_dotenv(env_path, override=True)
print(f"{'✓' if loaded else '×'} .env {'loaded' if loaded else 'NOT FOUND'} → {env_path}")

# ── Parse PROJECT_ENDPOINT ───────────────────────────────────────────────────
project_endpoint_url = os.getenv("PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")

if not project_endpoint_url:
    raise EnvironmentError(
        "PROJECT_ENDPOINT not set in .env.\n"
        "  Example: PROJECT_ENDPOINT=https://<hub>.services.ai.azure.com/api/projects/<project-name>"
    )

parsed        = urlparse(project_endpoint_url)
base_endpoint = f"{parsed.scheme}://{parsed.netloc}"
path_parts    = [p for p in parsed.path.split("/") if p]
project_name  = os.getenv("AZURE_AI_PROJECT_NAME") or (path_parts[-1] if path_parts else "")

# Hub resource name = first segment of the hostname (e.g. aifoundry-23252360)
hub_name = parsed.netloc.split(".")[0]

print(f"✓ Base endpoint:  {base_endpoint}")
print(f"✓ Project name:   {project_name}")
print(f"✓ Hub resource:   {hub_name}")

# ── Resolve subscription_id & resource_group ─────────────────────────────────
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group  = os.getenv("AZURE_RESOURCE_GROUP")

if not subscription_id or not resource_group:
    print("  AZURE_SUBSCRIPTION_ID / AZURE_RESOURCE_GROUP not in .env — auto-detecting...")
    try:
        mgmt_token = credential.get_token("https://management.azure.com/.default").token
        headers = {"Authorization": f"Bearer {mgmt_token}"}

        # List all accessible subscriptions
        subs = requests.get(
            "https://management.azure.com/subscriptions?api-version=2020-01-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        for sub in subs:
            sub_id = sub["subscriptionId"]
            # Search for the Foundry hub: type=Microsoft.CognitiveServices/accounts, name=hub_name
            resources = requests.get(
                f"https://management.azure.com/subscriptions/{sub_id}/resources"
                f"?$filter=name eq '{hub_name}' and "
                f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
                f"&api-version=2021-04-01",
                headers=headers, timeout=15
            ).json().get("value", [])

            if resources:
                # ARM resource ID: /subscriptions/<sub>/resourceGroups/<rg>/providers/...
                rg_from_id = resources[0]["id"].split("/")[4]
                subscription_id = subscription_id or sub_id
                resource_group  = resource_group  or rg_from_id
                break

        if not (subscription_id and resource_group):
            raise RuntimeError(f"Hub '{hub_name}' not found in any accessible subscription.")

        print(f"  ✓ Auto-detected subscription ID:  {subscription_id[:8]}...")
        print(f"  ✓ Auto-detected resource group:   {resource_group}")
        print(f"  Tip: add these to your .env to skip auto-detection next time:")
        print(f"       AZURE_SUBSCRIPTION_ID={subscription_id}")
        print(f"       AZURE_RESOURCE_GROUP={resource_group}")

    except Exception as e:
        raise EnvironmentError(
            f"Auto-detection failed: {e}\n"
            "Please add these to your .env file manually:\n"
            "  AZURE_SUBSCRIPTION_ID=<your-subscription-id>\n"
            "  AZURE_RESOURCE_GROUP=<your-resource-group>\n"
            "(find both in the Azure AI Foundry portal → your project → Overview)"
        ) from e
else:
    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

# ── Initialize AIProjectClient ───────────────────────────────────────────────
try:
    client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")

## Verify Access to Models and Connections
Let's verify we can access all the required models and connections specified in the [prerequisites](../README.md#-prerequisites).

We need to validate:
- GPT models (gpt-4o, gpt-4o-mini) for chat/completion
- Embedding model for vector search 
- [Grounding with Bing](https://learn.microsoft.com/en-us/azure/ai-services/agents/how-to/tools/bing-grounding?view=azure-python-preview&tabs=python&pivots=overview)
- Azure AI Search connection

This validation ensures we have all the components needed to build our AI applications.

In [ ]:
# List all connections in this project via REST API
# (client.connections.list() routes through MachineLearningServices/workspaces
#  which doesn't exist for new CognitiveServices-based Foundry projects)

from azure.identity import AzureCliCredential
from azure.core.exceptions import ClientAuthenticationError

print("====> Connections in your Foundry project:")

# Use AzureCliCredential explicitly (works better in notebooks than DefaultAzureCredential)
try:
    cli_credential = AzureCliCredential()
    _token = cli_credential.get_token("https://management.azure.com/.default").token
except ClientAuthenticationError as e:
    print(f"✗ Authentication failed: {str(e)[:100]}")
    print("\nTo fix this:")
    print("  1. Run the 1-authentication.ipynb notebook first")
    print("  2. Or re-login with: az login --tenant {tenant_id}")
    raise

_headers = {"Authorization": f"Bearer {_token}"}

for _api_ver in ["2025-04-01-preview", "2024-12-01-preview", "2024-10-01-preview"]:
    _url = (
        f"https://management.azure.com/subscriptions/{subscription_id}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{hub_name}"
        f"/projects/{project_name}/connections"
        f"?api-version={_api_ver}"
    )
    _r = requests.get(_url, headers=_headers, timeout=10)
    if _r.status_code == 200:
        _conns = _r.json().get("value", [])
        for conn in _conns:
            _props = conn.get("properties", {})
            print(f"  [{_props.get('category', 'Unknown')}] {conn['name']}")
        break
else:
    print("  (Unable to list connections — check your .env settings)")

## Validate Model and Search Connections
The cell below validates that we have properly provisioned and connected to:
1. Azure OpenAI models through our Azure OpenAI connection
2. Azure AI Search through our Azure AI Search connection

Both of these services will be essential for building our AI applications. The OpenAI models will provide the core language capabilities, while Azure AI Search will enable efficient information retrieval and knowledge base functionality.



In [1]:
# List all connections and check for specific types via REST API
# (client.connections.list() routes through MachineLearningServices/workspaces
#  which doesn't exist for new CognitiveServices-based Foundry projects)

from azure.identity import AzureCliCredential
from azure.core.exceptions import ClientAuthenticationError

print("\n====> All connections (debugging):")

# Use AzureCliCredential explicitly (works better in notebooks than DefaultAzureCredential)
try:
    cli_credential = AzureCliCredential()
    _token = cli_credential.get_token("https://management.azure.com/.default").token
except ClientAuthenticationError as e:
    print(f"✗ Authentication failed: {str(e)[:100]}")
    print("\nTo fix this:")
    print("  1. Run the 1-authentication.ipynb notebook first")
    print("  2. Or re-login with: az login --tenant {tenant_id}")
    raise

_headers = {"Authorization": f"Bearer {_token}"}
search_conn_id = ""
openai_conn_id = ""
found_any = False

for _api_ver in ["2025-04-01-preview", "2024-12-01-preview", "2024-10-01-preview"]:
    _url = (
        f"https://management.azure.com/subscriptions/{subscription_id}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{hub_name}"
        f"/projects/{project_name}/connections"
        f"?api-version={_api_ver}"
    )
    _r = requests.get(_url, headers=_headers, timeout=10)
    if _r.status_code == 200:
        found_any = True
        for conn in _r.json().get("value", []):
            _name = conn.get("name", "unknown")
            _cat = conn.get("properties", {}).get("category", "Unknown")
            print(f"  [{_cat}] {_name}")
            
            # Match connections by name or category
            _cat_lower = _cat.lower()
            _name_lower = _name.lower()
            
            # Azure AI Search detection
            if ("search" in _cat_lower and "ai" in _cat_lower) or "aisearch" in _name_lower:
                search_conn_id = conn["id"]
                print(f"    ✓ Matched as Azure AI Search")
                
            # Azure OpenAI detection  
            elif ("openai" in _cat_lower and "azure" in _cat_lower) or ("openai" in _name_lower):
                openai_conn_id = conn["id"]
                print(f"    ✓ Matched as Azure OpenAI")
        break

if not found_any:
    print("  (Unable to fetch connections — check your .env settings)")
else:
    print("\n====> Connection IDs found:")
    if not search_conn_id:
        print("  Azure AI Search: Not found - Please create an Azure AI Search connection")
    else:
        print(f"  ✓ Azure AI Search: {search_conn_id}")
        
    if not openai_conn_id:
        print("  ⓘ Azure OpenAI: Not explicitly required")
        print("    (Use the Foundry's built-in OpenAI connection from models)")

NameError: name 'client' is not defined